# LSA-Driven Anomaly Search

This notebook actively searches for anomaly candidates using the same architecture as the pipeline:

1. Normalize raw texts into an anomaly-focused view.
2. Build TF-IDF vectors.
3. Apply LSA (Truncated SVD) to obtain dense semantic vectors.
4. Run Isolation Forest and rank documents by anomaly score.

Lower scores mean more abnormal behavior.

Reader guide:

- This notebook is task-3 focused.
- I keep all ranking outputs in `data/results/`.
- I include both numeric ranking tables and short text snippets for qualitative inspection.

In [7]:
from pathlib import Path

from core.data_io import ArticleDataset
from anomaly_detection import TextAnomalyDetector
from exploration import build_anomaly_candidate_table, sample_top_anomaly_texts
from preprocessing import TextNormalizer, TextPreprocessor

## Load dataset

I load raw data in original row order so document IDs stay aligned with every exported table.

In [8]:
project_root_path = Path.cwd().parent
results_data_directory_path = project_root_path / "data" / "results"
results_data_directory_path.mkdir(parents=True, exist_ok=True)

articles_csv_path = project_root_path / "data" / "raw" / "articles.csv"
articles_data_frame = ArticleDataset(input_csv_path=articles_csv_path).load_articles()

document_ids = articles_data_frame["doc_id"].tolist()
raw_texts = articles_data_frame["text"].tolist()

articles_data_frame.head()

,doc_id,text
0,DOC_00001,"When I first saw this, I thought for a second ..."
1,DOC_00002,The difficulties of a high Isp OTV include: Lo...
2,DOC_00003,"Forwarded from Neal Ausman, Galileo Mission Di..."
3,DOC_00004,Sjogren's syndrome has been known to induce dr...
4,DOC_00005,"Yes, I want to concentrate on other developmen..."


## Build anomaly features with TF-IDF + LSA

Why this setup:

- Character n-grams capture unusual formatting, repetition, and symbol patterns.
- LSA reduces sparse noise and gives a dense space for robust anomaly scoring.

In [9]:
text_normalizer = TextNormalizer()
normalized_bundle = text_normalizer.normalize_for_both_tasks(raw_texts)

anomaly_preprocessor = TextPreprocessor(
    vectorization_model_name="tfidf_lsa_dense",
    max_features=30000,
    min_document_frequency=1,
    max_document_frequency=1.0,
    ngram_range=(3, 5),
    analyzer_mode="char_wb",
    dense_embedding_dimension=256,
    random_seed=42,
)

# Fit vectorizer + LSA on anomaly-oriented normalized text.
anomaly_feature_matrix = anomaly_preprocessor.fit_transform(normalized_bundle.anomaly_texts)
anomaly_feature_matrix.shape

(2164, 256)

## Score anomalies

I run Isolation Forest and create a ranked candidate table.

Interpretation:

- Top rows are strongest anomaly candidates.
- `predicted_anomaly` shows model binary prediction using configured contamination.

In [10]:
anomaly_detector = TextAnomalyDetector(contamination_ratio=0.02, random_seed=42)
anomaly_mask, anomaly_scores = anomaly_detector.run_detection(anomaly_feature_matrix)

anomaly_candidate_table = build_anomaly_candidate_table(
    document_ids=document_ids,
    anomaly_scores=anomaly_scores.tolist(),
    anomaly_mask=anomaly_mask.tolist(),
)
anomaly_candidate_table.to_csv(results_data_directory_path / "notebook_03_anomaly_candidates.csv", index=False)

anomaly_candidate_table.head(20)

,doc_id,score,predicted_anomaly,anomaly_rank
0,DOC_01099,-0.498501,True,1
1,DOC_00223,-0.498501,True,2
2,DOC_01449,-0.492151,True,3
3,DOC_01373,-0.490573,True,4
4,DOC_00192,-0.484717,True,5
5,DOC_01172,-0.484590,True,6
6,DOC_00604,-0.484518,True,7
7,DOC_00862,-0.483729,True,8
8,DOC_00932,-0.480816,True,9
9,DOC_01064,-0.480396,True,10


## Inspect top candidates

Use this section for qualitative report evidence.

I sample top-ranked documents with small snippets so the outlier pattern can be inspected manually.

In [11]:
top_anomaly_samples = sample_top_anomaly_texts(
    anomaly_candidate_table=anomaly_candidate_table,
    document_ids=document_ids,
    raw_texts=raw_texts,
    top_k=10,
)
top_anomaly_samples.to_csv(results_data_directory_path / "notebook_03_top_anomaly_samples.csv", index=False)

top_anomaly_samples[["anomaly_rank", "doc_id", "score", "predicted_anomaly", "snippet"]]

,anomaly_rank,doc_id,score,predicted_anomaly,snippet
0,1,DOC_01099,-0.498501,True,: Help!! I need code/package/whatever to take ...
1,2,DOC_00223,-0.498501,True,Help!! I need code/package/whatever to take 3-...
2,3,DOC_01449,-0.492151,True,<>Hismanal (astemizole) is most definitely lin...
3,4,DOC_01373,-0.490573,True,Hismanal (astemizole) is most definitely linke...
4,5,DOC_00192,-0.484717,True,+++ ++Once inflated the substance was no longe...
5,6,DOC_01172,-0.484590,True,: It is meaningless to compare one player's pl...
6,7,DOC_00604,-0.484518,True,+ +I love the idea of an inflatable 1-mile lon...
7,8,DOC_00862,-0.483729,True,Thanks for the information! I assume p is the ...
8,9,DOC_00932,-0.480816,True,-*----- These effects are a very real concern ...
9,10,DOC_01064,-0.480396,True,: I have one complaint for the cameramen doing...


## Optional: Create submission-style top-50 table

This helper block builds a deterministic top-50 candidate output in assignment format.

In [12]:
submission_top_50 = anomaly_candidate_table.head(50).copy()
submission_top_50["anomaly"] = submission_top_50.index + 1
submission_table = submission_top_50[["anomaly", "doc_id"]]
submission_table.to_csv(results_data_directory_path / "notebook_03_anomalies_top_50.csv", index=False)
submission_table.head()

,anomaly,doc_id
0,1,DOC_01099
1,2,DOC_00223
2,3,DOC_01449
3,4,DOC_01373
4,5,DOC_00192


### Files produced by this notebook

- `data/results/notebook_03_anomaly_candidates.csv`
- `data/results/notebook_03_top_anomaly_samples.csv`
- `data/results/notebook_03_anomalies_top_50.csv`

